<a href="https://colab.research.google.com/github/bharathh26/Project_Intership/blob/main/LogR_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/karolpiczak/ESC-50.git

Cloning into 'ESC-50'...
remote: Enumerating objects: 4199, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 4199 (delta 62), reused 34 (delta 34), pack-reused 4130 (from 1)
Receiving objects: 100% (4199/4199), 878.77 MiB | 34.74 MiB/s, done.
Resolving deltas: 100% (292/292), done.
Updating files: 100% (2011/2011), done.


In [ ]:
import pandas as pd

data = pd.read_csv("ESC-50/meta/esc50.csv")
data.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [ ]:
# Install librosa library for audio processing
!pip install librosa

In [ ]:
# Select only required sound categories
categories = ['chainsaw','engine','rain','crickets']

filtered_data = data[data['category'].isin(categories)]

# Display filtered dataset
filtered_data.head()

,filename,fold,target,category,esc10,src_file,take
24,1-116765-A-41.wav,1,41,chainsaw,True,116765,A
62,1-17367-A-10.wav,1,10,rain,True,17367,A
74,1-18527-A-44.wav,1,44,engine,False,18527,A
75,1-18527-B-44.wav,1,44,engine,False,18527,B
92,1-19898-A-41.wav,1,41,chainsaw,True,19898,A


In [ ]:
categories = ['chainsaw','engine','rain','crickets']

filtered_data = data[data['category'].isin(categories)].copy()

filtered_data['label'] = filtered_data['category'].apply(
    lambda x: 1 if x in ['chainsaw','engine'] else 0

    )

In [ ]:
import librosa
import numpy as np
import os

In [ ]:
def extract_features(file_path):

    # Load the audio file
    audio, sr = librosa.load(file_path, duration=5)

    # Extract MFCC features from audio
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=60)
    # Take the mean of MFCC features
    mfcc_scaled = np.mean(mfcc.T, axis=0)

    return mfcc_scaled

In [ ]:
features = []
labels = []

audio_path = "ESC-50/audio"

# Loop through dataset
for index, row in filtered_data.iterrows():

    file_name = row['filename']

    # Full path of audio file
    file_path = os.path.join(audio_path, file_name)

    # Extract features
    feature = extract_features(file_path)

    # Append feature and label
    features.append(feature)
    labels.append(row['label'])

# Convert to numpy arrays
X = np.array(features)
y = np.array(labels)

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Scale training data
X_train_scaled = scaler.fit_transform(X_train)

# Scale test data
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Increase iterations for better convergence
model = LogisticRegression(max_iter=2000,C=1)

# Train model on scaled data
model.fit(X_train_scaled, y_train)

LogisticRegression(C=1, max_iter=2000)

In [ ]:
predictions = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Print accuracy
print("Accuracy:", accuracy_score(y_test, predictions))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

# Detailed report
print("Classification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.78125
Confusion Matrix:
[[14  5]
 [ 2 11]]
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.74      0.80        19
           1       0.69      0.85      0.76        13

    accuracy                           0.78        32
   macro avg       0.78      0.79      0.78        32
weighted avg       0.80      0.78      0.78        32



In [ ]:
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, predictions)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[14  5]
 [ 2 11]]


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]

print("Confusion Matrix")
print(cm)

print("True Negative:", TN)
print("False Positive:", FP)
print("False Negative:", FN)
print("True Positive:", TP)

fpr = FP/(FP+TN)

print("False Positive Rate:", fpr*100)

Confusion Matrix
[[14  5]
 [ 2 11]]
True Negative: 14
False Positive: 5
False Negative: 2
True Positive: 11
False Positive Rate: 26.31578947368421
